# GlioGrade — Data Audit Notebook

Run this on Google Colab to get a full picture of every dataset folder on your Drive before training anything.

**What this does:**
- Scans each folder you list and counts patients, scan types, file counts
- Samples a few NIfTI files per folder to detect skull-stripping, normalization, volume shapes
- Checks for existing train/val/test splits
- Reads your metadata CSV and shows the real class distribution (crucial for understanding class imbalance)
- Scans any existing `.ipynb` files you point it to and extracts preprocessing-related code cells
- Prints a final summary of what folder is best to use for training

**Instructions:** Fill in `FOLDERS_TO_SCAN`, `NOTEBOOKS_TO_SCAN`, and `METADATA_CSV` in Cell 3, then Run All.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install nibabel tqdm -q

In [ ]:
# ── Cell 2: Imports & Drive mount ─────────────────────────────────────────────
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 3: ✏️  CONFIGURE THIS ─────────────────────────────────────────────────
# List every folder on your Drive that might contain GlioGrade data.
# Use the full path starting with /content/drive/MyDrive/...

FOLDERS_TO_SCAN = [
    # Examples — replace with your actual paths:
    # '/content/drive/MyDrive/UCSF_data/T2biasCollected',
    # '/content/drive/MyDrive/UCSF_data/raw',
    # '/content/drive/MyDrive/GlioGrade/preprocessed',
    # '/content/drive/MyDrive/GlioGrade/train',
]

# Path to your metadata CSV (labels file from UCSF-PDGM)
METADATA_CSV = '/content/drive/MyDrive/UCSF_data/UCSF-PDGM-metadata_v2.csv'  # or None

# Paths to any existing training/preprocessing notebooks on your Drive
NOTEBOOKS_TO_SCAN = [
    # '/content/drive/MyDrive/GlioGrade/training.ipynb',
    # '/content/drive/MyDrive/GlioGrade/preprocessing.ipynb',
]

# How many NIfTI files to sample per folder for detailed inspection
SAMPLE_SIZE = 5

print('Config set. Proceed to Cell 4.')

In [ ]:
# ── Cell 4: Helper functions ───────────────────────────────────────────────────

def infer_scan_type(filename):
    """Infer MRI scan type from filename."""
    f = filename.lower()
    if 't2_flair' in f or 't2flair' in f:   return 'T2_FLAIR'
    if 'flair' in f:                          return 'FLAIR'
    if 't2_bias' in f or 't2bias' in f:      return 'T2_bias'
    if 't1c_bias' in f or 't1cbias' in f:    return 'T1c_bias'
    if 't1c' in f:                            return 'T1c'
    if 't1_bias' in f or 't1bias' in f:      return 'T1_bias'
    if 't1' in f:                             return 'T1'
    if 't2' in f:                             return 'T2'
    if 'tumor_seg' in f or 'tumor_only' in f: return 'tumor_seg'
    if 'brain_seg' in f or 'parenchyma' in f: return 'brain_seg'
    if 'seg' in f or 'mask' in f:            return 'segmentation'
    if 'dwi' in f:                            return 'DWI'
    if 'asl' in f:                            return 'ASL'
    if 'swi' in f:                            return 'SWI'
    if 'adc' in f:                            return 'ADC'
    return 'unknown'

def infer_normalization(data):
    vmax, vmin = float(np.max(data)), float(np.min(data))
    nonzero = data[data != 0]
    if vmax > 1000:
        return f'raw  (range [{vmin:.0f}, {vmax:.0f}])'
    if 0.0 <= vmin and vmax <= 1.05:
        return f'min-max [0,1]'
    if len(nonzero) > 0 and abs(float(np.mean(nonzero))) < 3 and float(np.std(nonzero)) < 5:
        return f'z-score (~mean {np.mean(nonzero):.2f}, std {np.std(nonzero):.2f})'
    return f'other  (range [{vmin:.2f}, {vmax:.2f}])'

def inspect_nifti(filepath):
    """Load a NIfTI and return a dict of key stats."""
    try:
        img = nib.load(str(filepath))
        data = img.get_fdata()
        zero_frac = float(np.mean(data == 0))
        nonzero   = data[data != 0]
        return {
            'ok': True,
            'shape':        data.shape,
            'voxel_mm':     tuple(round(float(v), 2) for v in img.header.get_zooms()[:3]),
            'dtype':        str(data.dtype),
            'vmin':         round(float(np.min(data)), 4),
            'vmax':         round(float(np.max(data)), 4),
            'mean_nz':      round(float(np.mean(nonzero)), 4) if len(nonzero) else 0,
            'std_nz':       round(float(np.std(nonzero)),  4) if len(nonzero) else 0,
            'zero_frac':    round(zero_frac, 3),
            'skull_stripped': zero_frac > 0.30,
            'normalization': infer_normalization(data),
        }
    except Exception as e:
        return {'ok': False, 'error': str(e)}

def extract_patient_id(path_str):
    """Try to extract UCSF-PDGM patient ID from a file/folder name."""
    m = re.search(r'(?:UCSF[-_]PDGM[-_])?(\d{4})', str(path_str), re.IGNORECASE)
    return int(m.group(1)) if m else None

print('Helper functions ready.')

In [ ]:
# ── Cell 5: Scan each folder ───────────────────────────────────────────────────

SPLIT_KEYWORDS = {'train', 'val', 'validation', 'test', 'holdout'}

def scan_folder(folder_path, sample_size=5):
    folder = Path(folder_path)
    if not folder.exists():
        return {'exists': False, 'path': str(folder)}

    result = {
        'exists':          True,
        'path':            str(folder),
        'structure':       'flat',
        'split_detected':  False,
        'subfolders':      [],
        'n_nifti':         0,
        'scan_types':      {},
        'patient_ids':     set(),
        'other_files':     [],
        'samples':         [],
    }

    subdirs = [d for d in folder.iterdir() if d.is_dir()]
    subdir_names_lower = {d.name.lower() for d in subdirs}

    if SPLIT_KEYWORDS & subdir_names_lower:
        result['split_detected'] = True
        result['structure'] = 'split'
    elif subdirs:
        first_names = [d.name for d in subdirs[:5]]
        result['structure'] = 'per-patient' if any('ucsf' in n.lower() or n.isdigit() for n in first_names) else 'nested'

    result['subfolders'] = sorted([d.name for d in subdirs])

    # Collect all NIfTI files (rglob is slow on large drives — use iterdir first)
    nifti_files = list(folder.rglob('*.nii')) + list(folder.rglob('*.nii.gz'))
    result['n_nifti'] = len(nifti_files)

    for p in nifti_files:
        st = infer_scan_type(p.name)
        result['scan_types'][st] = result['scan_types'].get(st, 0) + 1
        pid = extract_patient_id(str(p))
        if pid:
            result['patient_ids'].add(pid)

    result['patient_ids'] = sorted(result['patient_ids'])
    result['n_patients']  = len(result['patient_ids'])

    # Interesting non-NIfTI files
    for glob in ['*.csv', '*.json', '*.ipynb', '*.pkl', '*.h5', '*.keras', '*.pt', '*.pth', '*.npy', '*.npz']:
        result['other_files'].extend(str(p) for p in folder.rglob(glob))

    # Sample NIfTI files for deep inspection
    for p in nifti_files[:sample_size]:
        info = inspect_nifti(p)
        info['filename']  = p.name
        info['scan_type'] = infer_scan_type(p.name)
        result['samples'].append(info)

    return result


print(f'Scanning {len(FOLDERS_TO_SCAN)} folder(s)...\n')
all_reports = {}

for fpath in FOLDERS_TO_SCAN:
    print(f'  → {fpath} ...')
    all_reports[fpath] = scan_folder(fpath, sample_size=SAMPLE_SIZE)

print('\nScan complete.')

In [ ]:
# ── Cell 6: Print folder reports ──────────────────────────────────────────────

SEP  = '═' * 72
SEP2 = '─' * 72

print(SEP)
print('  FOLDER SCAN RESULTS')
print(SEP)

for fpath, r in all_reports.items():
    print(f'\n{SEP2}')
    print(f'  📁  {fpath}')
    print(f'{SEP2}')

    if not r['exists']:
        print('  ❌  Folder not found — check path above.')
        continue

    split_icon = '✅' if r['split_detected'] else '⚠️ '
    split_msg  = 'train/val/test split DETECTED' if r['split_detected'] else 'No split detected (raw flat folder)'

    print(f"  Structure:    {r['structure']}")
    print(f"  NIfTI files:  {r['n_nifti']}")
    print(f"  Patients:     {r['n_patients']} unique IDs detected")
    if r['patient_ids']:
        sample_ids = r['patient_ids'][:5]
        print(f"  Sample IDs:   {sample_ids}{'...' if len(r['patient_ids']) > 5 else ''}")
    print(f"  {split_icon}  {split_msg}")

    if r['subfolders']:
        shown = r['subfolders'][:8]
        print(f"  Subfolders:   {shown}{'...' if len(r['subfolders']) > 8 else ''}")

    if r['scan_types']:
        print(f"\n  Scan types:")
        for stype, cnt in sorted(r['scan_types'].items(), key=lambda x: -x[1]):
            bar = '█' * min(cnt, 40)
            print(f"    {stype:<22} {cnt:>4}  {bar}")

    if r['other_files']:
        print(f"\n  Other files of interest ({len(r['other_files'])}) :")
        for f in r['other_files'][:10]:
            print(f"    {f}")
        if len(r['other_files']) > 10:
            print(f"    ... and {len(r['other_files'])-10} more")

    if r['samples']:
        print(f"\n  NIfTI sample inspection (first {len(r['samples'])}):")
        for s in r['samples']:
            if not s.get('ok', False):
                print(f"    ⚠️  {s.get('filename','?')}  →  ERROR: {s.get('error')}")
                continue
            skull = '✅ skull-stripped' if s['skull_stripped'] else '⚠️  NOT stripped (low zero fraction)'
            print(f"    {s['filename']}")
            print(f"      Type: {s['scan_type']:<14}  Shape: {str(s['shape']):<22}  Voxel: {s['voxel_mm']} mm")
            print(f"      Norm: {s['normalization']:<35}  zeros: {s['zero_frac']:.1%}  → {skull}")

print(f'\n{SEP}')

In [ ]:
# ── Cell 7: Metadata CSV — class distribution ──────────────────────────────────

print(SEP)
print('  METADATA CSV — CLASS DISTRIBUTION')
print(SEP)

if METADATA_CSV is None or not Path(METADATA_CSV).exists():
    print(f'  ⚠️  Metadata CSV not found.  Set METADATA_CSV in Cell 3.')
else:
    meta = pd.read_csv(METADATA_CSV)
    print(f'  Path:     {METADATA_CSV}')
    print(f'  Shape:    {meta.shape}')
    print(f'  Columns:  {list(meta.columns)}\n')

    # ── Type distribution
    diag_col = next((c for c in meta.columns if 'pathologic' in c.lower() or 'diagnosis' in c.lower()), None)
    if diag_col:
        dist = meta[diag_col].value_counts()
        print(f'  Type distribution  ("{diag_col}"):')
        for label, cnt in dist.items():
            bar = '█' * min(int(cnt / 5), 40)
            print(f"    {str(label):<58} {cnt:>4}  ({100*cnt/len(meta):.1f}%)  {bar}")

        # Known WHO 2021 labels
        known = {
            'Glioblastoma, IDH-wildtype',
            'Astrocytoma, IDH-wildtype',
            'Oligodendroglioma, IDH-mutant, 1p/19q-codeleted',
            'Astrocytoma, IDH-mutant'
        }
        misc_mask = ~meta[diag_col].isin(known)
        misc = meta.loc[misc_mask, diag_col].value_counts()
        if len(misc) > 0:
            print(f'\n  ⚠️  Labels outside the 4 known WHO 2021 classes (will be dropped):')
            for lbl, cnt in misc.items():
                print(f'    "{lbl}": {cnt}')
        else:
            print('\n  ✅  All labels match the 4 WHO 2021 classes.')

    # ── Grade distribution
    grade_col = next((c for c in meta.columns if c.lower() in ('grade', 'who grade', 'tumor grade')), None)
    if grade_col:
        print(f'\n  Grade distribution  ("{grade_col}"):')
        gdist = meta[grade_col].value_counts().sort_index()
        for g, cnt in gdist.items():
            bar = '█' * min(int(cnt / 5), 40)
            print(f'    Grade {g}: {cnt:>4}  ({100*cnt/len(meta):.1f}%)  {bar}')

    # ── Class imbalance warning
    print('\n  ⚠️  Class imbalance analysis for TRAINING:')
    print('  A plain cross-entropy model will likely learn to predict the majority class.')
    print('  Majority-class baseline accuracies (what "doing nothing" achieves):')
    if diag_col:
        dist_vals = meta[meta[diag_col].isin(known)][diag_col].value_counts()
        total = dist_vals.sum()
        print(f'    Typing  →  {dist_vals.iloc[0] / total:.1%} (majority: "{dist_vals.index[0]}")')
    if grade_col:
        gdist_vals = meta[grade_col].value_counts()
        total = gdist_vals.sum()
        print(f'    Grading →  {gdist_vals.iloc[0] / total:.1%} (majority: Grade {gdist_vals.index[0]})')

    # ── Cross-reference: how many patients have scan files?
    all_found_ids = set()
    for r in all_reports.values():
        if r.get('exists'):
            all_found_ids.update(r.get('patient_ids', []))

    if 'ID' in meta.columns and all_found_ids:
        meta_ids = set(meta['ID'].str.replace('UCSF-PDGM-', '', regex=False).astype(int))
        overlap  = meta_ids & all_found_ids
        meta_only = meta_ids - all_found_ids
        files_only = all_found_ids - meta_ids
        print(f'\n  Cross-reference  (metadata  ↔  scan files found):')
        print(f'    Patients in metadata:         {len(meta_ids)}')
        print(f'    Patient IDs found in folders: {len(all_found_ids)}')
        print(f'    ✅  In both (usable):           {len(overlap)}')
        if meta_only:
            print(f'    ⚠️   In metadata only (no scan): {len(meta_only)}  e.g. {sorted(meta_only)[:5]}')
        if files_only:
            print(f'    ⚠️   Scan file but no label:     {len(files_only)}  e.g. {sorted(files_only)[:5]}')

print(f'\n{SEP}')

In [ ]:
# ── Cell 8: Visualise sample slices from each folder ──────────────────────────
# Plots one middle axial slice from each sampled NIfTI per folder.
# Helps you visually confirm preprocessing state.

for fpath, r in all_reports.items():
    if not r.get('exists') or not r.get('samples'):
        continue

    good_samples = [s for s in r['samples'] if s.get('ok')]
    if not good_samples:
        continue

    n = len(good_samples)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    fig.suptitle(f'Middle axial slices\n{Path(fpath).name}', fontsize=11)

    nifti_paths = list(Path(fpath).rglob('*.nii')) + list(Path(fpath).rglob('*.nii.gz'))

    for ax, (s, nii_path) in zip(axes, zip(good_samples, nifti_paths[:n])):
        try:
            data = nib.load(str(nii_path)).get_fdata()
            mid  = data.shape[2] // 2
            ax.imshow(np.rot90(data[:, :, mid]), cmap='gray', aspect='auto')
            skull_tag = '(stripped)' if s['skull_stripped'] else '(raw)'
            ax.set_title(f"{s['scan_type']}\n{s['shape']} {skull_tag}", fontsize=8)
        except Exception as e:
            ax.set_title(f'Error: {e}', fontsize=7)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 9: Scan existing notebooks for preprocessing code ────────────────────

PREP_KEYWORDS = [
    'skull', 'strip', ' bet ', 'bias', 'n4', 'normaliz', 'resize',
    'resample', 'augment', 'train_test', 'stratif', 'label_map',
    'preprocessing', 'nib.load', 'get_fdata', 'sitk', 'ants',
    'zscore', 'z_score', 'minmax', 'min_max', 'weighted', 'class_weight',
    'batch_size', 'epochs', 'optimizer', 'model.fit', 'model.compile',
    'Conv3D', 'MaxPool3D', 'BatchNorm', 'DataLoader'
]

def load_notebook_cells(path):
    try:
        with open(str(path), 'r', encoding='utf-8') as f:
            nb = json.load(f)
        cells = []
        for i, cell in enumerate(nb.get('cells', [])):
            if cell['cell_type'] == 'code':
                src = ''.join(cell.get('source', []))
                if src.strip():
                    cells.append({'index': i, 'source': src})
        return cells
    except Exception as e:
        return [{'index': 0, 'source': f'ERROR reading notebook: {e}'}]

def is_preprocessing_cell(source):
    low = source.lower()
    return any(kw in low for kw in PREP_KEYWORDS)

# Also auto-detect .ipynb in scanned folders
auto_nbs = []
for r in all_reports.values():
    if r.get('exists'):
        auto_nbs.extend(f for f in r.get('other_files', []) if f.endswith('.ipynb'))

all_nb_paths = list(dict.fromkeys(NOTEBOOKS_TO_SCAN + auto_nbs))  # deduplicate, preserve order

print(SEP)
print('  EXISTING NOTEBOOK SCAN — PREPROCESSING & TRAINING STEPS')
print(SEP)

if not all_nb_paths:
    print('\n  No notebooks found.')
    print('  Add paths to NOTEBOOKS_TO_SCAN in Cell 3, or they will be auto-detected')
    print('  if they live inside one of the FOLDERS_TO_SCAN directories.')
else:
    for nb_path in all_nb_paths:
        p = Path(nb_path)
        print(f'\n  📓 {p.name}')
        print(f'     Full path: {nb_path}')

        cells = load_notebook_cells(nb_path)
        print(f'     Total code cells: {len(cells)}')

        prep_cells = [c for c in cells if is_preprocessing_cell(c['source'])]
        print(f'     Relevant cells found: {len(prep_cells)}')

        for c in prep_cells:
            lines = c['source'].strip().split('\n')
            preview = '\n'.join(f'       {l}' for l in lines[:12])
            print(f'\n     ── Cell #{c["index"]} ──────────────────────')
            print(preview)
            if len(lines) > 12:
                print(f'       ... ({len(lines) - 12} more lines)')

print(f'\n{SEP}')

In [ ]:
# ── Cell 10: Final summary ─────────────────────────────────────────────────────

print(SEP)
print('  FINAL SUMMARY')
print(SEP)

all_patient_ids = set()
for r in all_reports.values():
    if r.get('exists'):
        all_patient_ids.update(r.get('patient_ids', []))

print(f'\n  Folders scanned:           {len(FOLDERS_TO_SCAN)}')
print(f'  Folders found/accessible:  {sum(1 for r in all_reports.values() if r.get("exists"))}')
print(f'  Total unique patient IDs:  {len(all_patient_ids)}')

print('\n  Per-folder status:')
for fpath, r in all_reports.items():
    if not r.get('exists'):
        status = '❌ NOT FOUND'
    else:
        parts = []
        parts.append(f"{r['n_nifti']} NIfTI files")
        parts.append(f"{r['n_patients']} patients")
        good_samples = [s for s in r.get('samples', []) if s.get('ok')]
        if good_samples:
            avg_zero = np.mean([s['zero_fraction'] for s in good_samples])
            parts.append('skull-stripped ✅' if avg_zero > 0.3 else 'NOT stripped ⚠️')
        parts.append('split ✅' if r.get('split_detected') else 'no split ⚠️')
        status = '  |  '.join(parts)
    short = str(Path(fpath).name)
    print(f'    {short:<35}  {status}')

print('\n  ─────────────────────────────────────────────────────────────────────')
print('  ACTION ITEMS  (fill in manually after reviewing above):')
print('  ─────────────────────────────────────────────────────────────────────')
print('  [ ] Best preprocessed folder to use for training:  _______________')
print('  [ ] Does it need skull-stripping?                   yes / no')
print('  [ ] Does it need bias correction?                   yes / no')
print('  [ ] Does it need a train/val/test split?            yes / no')
print('  [ ] What scan type to use?                          _______________')
print('  [ ] Confirmed metadata CSV has matching IDs?        yes / no')
print('  [ ] Any misc labels to drop from metadata?          yes / no')
print(f'\n{SEP}')